In [1]:
import pandas as pd
import numpy as np


## Remplissage des NA

In [2]:
df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')

#### On repart du df_ventes sans les colonnes avec trop de variables manquantes

In [3]:
valeurs_manquantes = df_ventes.isna().sum()/df_ventes.shape[0]
valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)
valeurs_manquantes_list

['prix_maison',
 'prix_terrain',
 'parking',
 'nb_terraces',
 'videophone',
 'porte_digicode',
 'surface_balcon']

In [4]:
df_ventes_filtered = df_ventes.drop(valeurs_manquantes_list, axis = 1)
df_ventes_filtered.shape

(27737, 51)

In [5]:
column_n6 = [ column for column in df_ventes_filtered.columns if "n6" in column]
column_n6

['loyer_m2_median_n6', 'nb_log_n6', 'taux_rendement_n6']

#### Précédemment nous avions dit que nous gardions plutôt les varaibles suffixées _n7 basées sur un plus grand nombre de logements 

In [6]:
df_ventes_filtered = df_ventes_filtered.drop(column_n6, axis = 1)

In [7]:
df_ventes_filtered.shape

(27737, 48)

In [8]:
## On va beaucoups'appuyer sur la colonne typedebien

In [9]:
df_ventes_filtered['typedebien'].value_counts()

typedebien
m     13288
a     13158
mn      650
an      640
l         1
Name: count, dtype: int64

In [10]:
## On enlève le seul l (lot?) et on met tous les appartements en a et maison en m

In [11]:
df_ventes_filtered = df_ventes_filtered[df_ventes_filtered['typedebien'] != 'l']
df_ventes_filtered.shape

(27736, 48)

In [12]:
df_ventes_filtered['typedebien'] = df_ventes_filtered['typedebien'].replace('an', 'a')
df_ventes_filtered['typedebien'] = df_ventes_filtered['typedebien'].replace('mn', 'm')
df_ventes_filtered['typedebien'].value_counts()

typedebien
m    13938
a    13798
Name: count, dtype: int64

In [13]:
df_ventes_filtered.shape

(27736, 48)

In [14]:
df_ventes_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 48 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           27736 non-null  object 
 1   typedebien               27736 non-null  object 
 2   typedetransaction        27736 non-null  object 
 3   etage                    27736 non-null  int64  
 4   surface                  27736 non-null  int64  
 5   surface_terrain          11444 non-null  float64
 6   nb_pieces                27736 non-null  int64  
 7   prix_bien                27736 non-null  int64  
 8   mensualiteFinance        27736 non-null  int64  
 9   balcon                   27736 non-null  int64  
 10  eau                      27736 non-null  int64  
 11  bain                     27736 non-null  int64  
 12  dpeL                     27736 non-null  object 
 13  dpeC                     17150 non-null  float64
 14  ma

In [15]:
valeurs_manq_resid = df_ventes_filtered.isna().sum()
valeurs_manq_resid_list = list(valeurs_manq_resid[valeurs_manq_resid > 0].index)
valeurs_manq_resid_list

['surface_terrain',
 'dpeC',
 'nb_etages',
 'places_parking',
 'cave',
 'ges_class',
 'annee_construction',
 'nb_toilettes',
 'ascenseur',
 'nb_logements_copro',
 'charges_copro',
 'chauffage_energie',
 'chauffage_systeme',
 'chauffage_mode',
 'logement_neuf',
 'duree_int',
 'loyer_m2_median_n7',
 'nb_log_n7',
 'taux_rendement_n7']

In [16]:
print(f"Nombre de colonnes avec données manquantes : {len(valeurs_manq_resid_list)}")

Nombre de colonnes avec données manquantes : 19


In [17]:
valeurs_manq_resid_list_quanti = df_ventes_filtered[valeurs_manq_resid_list].select_dtypes(exclude='object').columns
valeurs_manq_resid_list_quanti

Index(['surface_terrain', 'dpeC', 'nb_etages', 'places_parking',
       'annee_construction', 'nb_toilettes', 'nb_logements_copro',
       'charges_copro', 'duree_int', 'loyer_m2_median_n7', 'nb_log_n7',
       'taux_rendement_n7'],
      dtype='object')

In [18]:
valeurs_manq_resid_list_quali = df_ventes_filtered[valeurs_manq_resid_list].select_dtypes(include='object').columns
valeurs_manq_resid_list_quali

Index(['cave', 'ges_class', 'ascenseur', 'chauffage_energie',
       'chauffage_systeme', 'chauffage_mode', 'logement_neuf'],
      dtype='object')

In [19]:
print(f"Nombre de colonnes avec données manquantes quanti: {len(valeurs_manq_resid_list_quanti)}\nNombre de colonnes avec données manquantes quanti: {len(valeurs_manq_resid_list_quali)}")

Nombre de colonnes avec données manquantes quanti: 12
Nombre de colonnes avec données manquantes quanti: 7


#### Remplir d'abord les variables quanti

#### On décide de faire une imputation via les plus proches voisins mais en se focalisant sur les biens de mêmes types étant roche au niveau localication et nombre de pièces


In [20]:
neighbours_columns = ["nb_pieces", "mapCoordonneesLatitude", "mapCoordonneesLongitude"]
type_column = 'typedebien'

In [21]:
from sklearn.impute import KNNImputer

In [22]:
def df_imputed(input_df: pd.DataFrame):
    imputer = KNNImputer()
    cols_backup = input_df.columns
    index_backup = input_df.index
    imputed_data = imputer.fit_transform(input_df)  
    return pd.DataFrame(data=imputed_data, columns=cols_backup, index=index_backup)
    

In [ ]:
for interested_column in valeurs_manq_resid_list_quanti:
    print(f"Processing column {interested_column}")
    # On se focalise sur la colonne et nos colonnes voisines
    df_extract = df_ventes_filtered[[type_column] + [interested_column] + neighbours_columns]
    # On distingue par type de bien
    df_extract_app = df_extract[df_extract[type_column] =='a'][[interested_column] + neighbours_columns]
    df_extract_mais = df_extract[df_extract[type_column] =='m'][[interested_column] + neighbours_columns]
    
    df_filled_app = df_imputed(df_extract_app)
    df_filled_mais = df_imputed(df_extract_mais)
    df_filled = pd.concat([df_filled_app, df_filled_mais])

    df_ventes_filtered[interested_column] = df_filled[interested_column]

Processing column surface_terrain
Processing column dpeC
Processing column nb_etages
Processing column places_parking
Processing column annee_construction
Processing column nb_toilettes
Processing column nb_logements_copro
Processing column charges_copro


In [141]:
df_ventes_filtered[valeurs_manq_resid_list_quanti].info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   surface_terrain     27736 non-null  float64
 1   dpeC                27736 non-null  float64
 2   nb_etages           27736 non-null  float64
 3   places_parking      27736 non-null  float64
 4   annee_construction  27736 non-null  float64
 5   nb_toilettes        27736 non-null  float64
 6   nb_logements_copro  27736 non-null  float64
 7   charges_copro       27736 non-null  float64
 8   duree_int           27736 non-null  float64
 9   loyer_m2_median_n7  27736 non-null  float64
 10  nb_log_n7           27736 non-null  float64
 11  taux_rendement_n7   27736 non-null  float64
dtypes: float64(12)
memory usage: 3.8+ MB


In [142]:
df_ventes_filtered.to_csv("df_filled_quanti.csv")